In [1]:
import sys, os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "src"))

import pandas as pd
import matplotlib.pyplot as plt

from data_cleaning import (
    load_raw, add_flags, fill_missing_metadata,
    fill_discount_bruto, fill_residual_discount_bruto
)

## 1. Data Loading and Cleaning

Dataset: 283,533 transactions, 6 SKUs, 12 warehouses, ~25 months (Jan 2025 - Jan 2027).

Cleaning rules are implemented in `src/data_cleaning.py` and summarized step by step below.

In [2]:
df = load_raw("../data/20260806_prueba_tecnica_dataset.csv")
print(f"Total rows: {len(df)}")

df = add_flags(df)
print(f"Cancelled tickets: {df['is_cancelled'].sum()}")
print(f"Gifts/samples: {df['is_gift'].sum()}")

df = fill_missing_metadata(df)
df = fill_discount_bruto(df)
df = fill_residual_discount_bruto(df)

print("\nRemaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Total rows: 283533
Cancelled tickets: 500
Gifts/samples: 500

Remaining nulls:
id_combo        215544
combo           215544
bruto                1
product_cost       110
dtype: int64


**Decision:** both row types are kept (not dropped), marked with a flag.
- `is_cancelled`: excluded from any demand aggregation (Challenge A).
- `is_gift`: counts as physical demand (Challenge A), but excluded from
  effective price / elasticity calculations (Challenge B), since unit_price=0
  is not a real market price.

**Decision:** the row with missing metadata (category, subcategory, brand,
basket) was filled by mapping from other rows with the same `product_code`,
since these fields are constant per SKU. No row was dropped.

**Decision:** `discount`, `bruto`, and `sell_in_amount` are algebraically
related: `bruto = sell_in_amount / (1 - discount)`. Most missing values were
resolved by deriving one column from the other two. Organic sales
(`id_combo` null) with a remaining null default to `discount = 0`.

For 3 rows that belonged to a promotion but had both values missing,
`discount` was imputed with the average discount of that same `id_combo`.
One row remains null in `bruto`: a gift transaction (`amount=0`,
`discount=1.0`) where the formula is mathematically undefined (division by
zero) — left null rather than forcing an arbitrary value.

In [3]:
from data_cleaning import fill_product_cost

df = fill_product_cost(df)
print("Remaining nulls in product_cost:", df["product_cost"].isnull().sum())
print("Imputed rows:", df["product_cost_imputed"].sum())

Remaining nulls in product_cost: 0
Imputed rows: 110
